In [1]:
import os
import uuid
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Concatenate, Input, Conv2D, MaxPooling2D, Flatten, Dropout
from tensorflow.keras.applications import ResNet50, VGG16, InceptionV3, DenseNet121, MobileNetV2
from tensorflow.keras.optimizers import Adam
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import cohen_kappa_score, matthews_corrcoef, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.preprocessing import label_binarize
import joblib
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime

# Set random seed for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Define constants
DATA_DIR = r"C:\Users\KISHOR\Downloads\MSID Classification\Monkeypox Skin Image Dataset"
CATEGORIES = ["Normal", "Monkeypox", "Chickenpox", "Measles"]
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 50
NUM_CLASSES = len(CATEGORIES)
VALIDATION_SPLIT = 0.2
LEARNING_RATE = 0.001
MODEL_NAMES = ['DeepLearning+ML', 'ResNet+VGG+Inception', 'DenseNet+MobileNet+AlexNet']

# Step 1: Data Preprocessing
def load_data():
    train_datagen = ImageDataGenerator(
        rescale=1./255,
        validation_split=VALIDATION_SPLIT,
        rotation_range=30,
        zoom_range=0.2,
        width_shift_range=0.3,
        height_shift_range=0.3,
        shear_range=0.2,
        horizontal_flip=True,
        vertical_flip=True,
        fill_mode="nearest"
    )

    validation_datagen = ImageDataGenerator(
        rescale=1./255,
        validation_split=VALIDATION_SPLIT
    )

    train_generator = train_datagen.flow_from_directory(
        DATA_DIR,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        subset='training',
        shuffle=True
    )

    validation_generator = validation_datagen.flow_from_directory(
        DATA_DIR,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        subset='validation',
        shuffle=False
    )

    return train_generator, validation_generator

# Step 2: Define Fusion Models
# Model 1: Deep Learning + ML Fusion Model
def create_model_1():
    input_layer = Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = Conv2D(64, (3, 3), activation='relu', padding='same')(input_layer)
    x = MaxPooling2D((2, 2))(x)
    x = Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = MaxPooling2D((2, 2))(x)
    x = Conv2D(256, (3, 3), activation='relu', padding='same')(x)
    x = MaxPooling2D((2, 2))(x)
    x = Conv2D(512, (3, 3), activation='relu', padding='same')(x)
    x = MaxPooling2D((2, 2))(x)
    x = Flatten()(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.5)(x)
    cnn_features = Dense(128, activation='relu')(x)

    model = Model(inputs=input_layer, outputs=cnn_features)
    rf_classifier = RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    )
    
    return model, rf_classifier

# Model 2: ResNet50 + VGG16 + InceptionV3 Fusion
def create_model_2():
    resnet = ResNet50(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
    vgg = VGG16(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
    inception = InceptionV3(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))

    for layer in resnet.layers:
        layer.trainable = False
    for layer in vgg.layers:
        layer.trainable = False
    for layer in inception.layers:
        layer.trainable = False

    input_layer = Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    resnet_out = resnet(input_layer)
    vgg_out = vgg(input_layer)
    inception_out = inception(input_layer)

    resnet_out = GlobalAveragePooling2D()(resnet_out)
    vgg_out = GlobalAveragePooling2D()(vgg_out)
    inception_out = GlobalAveragePooling2D()(inception_out)

    combined = Concatenate()([resnet_out, vgg_out, inception_out])
    x = Dense(1024, activation='relu')(combined)
    x = Dropout(0.5)(x)
    x = Dense(512, activation='relu')(x)
    x = Dropout(0.3)(x)
    output = Dense(NUM_CLASSES, activation='softmax')(x)

    model = Model(inputs=input_layer, outputs=output)
    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# Model 3: DenseNet121 + MobileNetV2 + AlexNet Fusion
def create_model_3():
    densenet = DenseNet121(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
    mobilenet = MobileNetV2(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))

    for layer in densenet.layers:
        layer.trainable = False
    for layer in mobilenet.layers:
        layer.trainable = False

    input_layer = Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    
    # AlexNet Implementation
    x = Conv2D(96, (11, 11), strides=4, activation='relu', padding='valid')(input_layer)
    x = MaxPooling2D((3, 3), strides=2)(x)
    x = Conv2D(256, (5, 5), padding='same', activation='relu')(x)
    x = MaxPooling2D((3, 3), strides=2)(x)
    x = Conv2D(384, (3, 3), padding='same', activation='relu')(x)
    x = Conv2D(384, (3, 3), padding='same', activation='relu')(x)
    x = Conv2D(256, (3, 3), padding='same', activation='relu')(x)
    x = MaxPooling2D((3, 3), strides=2)(x)
    alex_out = GlobalAveragePooling2D()(x)

    densenet_out = densenet(input_layer)
    mobilenet_out = mobilenet(input_layer)
    densenet_out = GlobalAveragePooling2D()(densenet_out)
    mobilenet_out = GlobalAveragePooling2D()(mobilenet_out)

    combined = Concatenate()([densenet_out, mobilenet_out, alex_out])
    x = Dense(1024, activation='relu')(combined)
    x = Dropout(0.5)(x)
    x = Dense(512, activation='relu')(x)
    x = Dropout(0.3)(x)
    output = Dense(NUM_CLASSES, activation='softmax')(x)

    model = Model(inputs=input_layer, outputs=output)
    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# Step 3: Evaluation Metrics
def evaluate_model(y_true, y_pred, y_prob):
    metrics = {}
    metrics['Cohen Kappa'] = cohen_kappa_score(y_true, y_pred)
    metrics['MCC'] = matthews_corrcoef(y_true, y_pred)
    metrics['Precision'] = precision_score(y_true, y_pred, average='weighted')
    metrics['Recall'] = recall_score(y_true, y_pred, average='weighted')
    metrics['F1'] = f1_score(y_true, y_pred, average='weighted')
    metrics['AUC-ROC'] = roc_auc_score(label_binarize(y_true, classes=[0,1,2,3]), y_prob, multi_class='ovr')
    
    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    return metrics, cm

# Step 4: ROC Curve Data
def get_roc_data(y_true, y_prob):
    roc_data = {}
    y_bin = label_binarize(y_true, classes=[0,1,2,3])
    for i in range(NUM_CLASSES):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
        roc_data[CATEGORIES[i]] = {'fpr': fpr, 'tpr': tpr, 'auc': roc_auc_score(y_bin[:, i], y_prob[:, i])}
    return roc_data

# Step 5: Training and Evaluation
def train_and_evaluate():
    train_generator, validation_generator = load_data()
    results = {}
    confusion_matrices = {}
    roc_curves = {}
    models = {
        'DeepLearning+ML': create_model_1(),
        'ResNet+VGG+Inception': create_model_2(),
        'DenseNet+MobileNet+AlexNet': create_model_3()
    }

    best_model = None
    best_f1 = 0
    best_model_name = ""

    for model_name, model in models.items():
        print(f"Training {model_name}...")
        if model_name == 'DeepLearning+ML':
            cnn_model, rf_model = model
            # Extract features for RandomForest
            train_features = []
            train_labels = []
            for i in range(len(train_generator)):
                batch_x, batch_y = train_generator[i]
                features = cnn_model.predict(batch_x, verbose=0)
                train_features.append(features)
                train_labels.append(np.argmax(batch_y, axis=1))
            train_features = np.vstack(train_features)
            train_labels = np.hstack(train_labels)
            rf_model.fit(train_features, train_labels)
            
            val_features = []
            val_true = []
            for i in range(len(validation_generator)):
                batch_x, batch_y = validation_generator[i]
                features = cnn_model.predict(batch_x, verbose=0)
                val_features.append(features)
                val_true.append(np.argmax(batch_y, axis=1))
            val_features = np.vstack(val_features)
            val_true = np.hstack(val_true)
            
            val_pred = rf_model.predict(val_features)
            val_prob = rf_model.predict_proba(val_features)
        else:
            model.fit(
                train_generator,
                epochs=EPOCHS,
                validation_data=validation_generator,
                verbose=1
            )
            val_pred = np.argmax(model.predict(validation_generator, verbose=0), axis=1)
            val_prob = model.predict(validation_generator, verbose=0)
            val_true = validation_generator.classes

        metrics, cm = evaluate_model(val_true, val_pred, val_prob)
        roc_data = get_roc_data(val_true, val_prob)
        results[model_name] = metrics
        confusion_matrices[model_name] = cm
        roc_curves[model_name] = roc_data

        # Save the best model based on F1 score
        if metrics['F1'] > best_f1:
            best_f1 = metrics['F1']
            best_model = model
            best_model_name = model_name
            if model_name == 'DeepLearning+ML':
                cnn_model.save('best_model_cnn.h5')
                joblib.dump(rf_model, 'best_model_rf.pkl')
            else:
                model.save('best_model.h5')

    return results, confusion_matrices, roc_curves, best_model, best_model_name

# Step 6: Visualizations with Plotly
def visualize_results(results, confusion_matrices, roc_curves):
    # Individual Model Visualizations (10 per model)
    for model_name in results:
        metrics = results[model_name]
        cm = confusion_matrices[model_name]
        roc_data = roc_curves[model_name]
        
        # Visualization 1: Bar Plot of Metrics
        fig1 = go.Figure()
        for metric, value in metrics.items():
            fig1.add_trace(go.Bar(x=[metric], y=[value], name=metric, marker_color='royalblue'))
        fig1.update_layout(
            title=f"{model_name}: Metrics Bar Plot",
            xaxis_title="Metric",
            yaxis_title="Score",
            template='plotly_dark'
        )
        fig1.show()

        # Visualization 2: Radar Plot of Metrics
        fig2 = go.Figure()
        fig2.add_trace(go.Scatterpolar(
            r=list(metrics.values()),
            theta=list(metrics.keys()),
            fill='toself',
            name=model_name,
            line_color='cyan'
        ))
        fig2.update_layout(
            title=f"{model_name}: Metrics Radar Plot",
            polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
            template='plotly_dark'
        )
        fig2.show()

        # Visualization 3: Line Plot of Metrics
        fig3 = go.Figure()
        fig3.add_trace(go.Scatter(
            x=list(metrics.keys()),
            y=list(metrics.values()),
            mode='lines+markers',
            name=model_name,
            line=dict(color='orange')
        ))
        fig3.update_layout(
            title=f"{model_name}: Metrics Line Plot",
            xaxis_title="Metric",
            yaxis_title="Score",
            template='plotly_dark'
        )
        fig3.show()

        # Visualization 4: Confusion Matrix Heatmap
        fig4 = px.imshow(
            cm,
            labels=dict(x="Predicted", y="True", color="Count"),
            x=CATEGORIES,
            y=CATEGORIES,
            text_auto=True,
            color_continuous_scale='Blues'
        )
        fig4.update_layout(
            title=f"{model_name}: Confusion Matrix",
            template='plotly_dark'
        )
        fig4.show()

        # Visualization 5: ROC Curves
        fig5 = go.Figure()
        for category in roc_data:
            fig5.add_trace(go.Scatter(
                x=roc_data[category]['fpr'],
                y=roc_data[category]['tpr'],
                mode='lines',
                name=f"{category} (AUC={roc_data[category]['auc']:.2f})"
            ))
        fig5.add_trace(go.Scatter(
            x=[0, 1], y=[0, 1],
            mode='lines',
            line=dict(dash='dash', color='gray'),
            showlegend=False
        ))
        fig5.update_layout(
            title=f"{model_name}: ROC Curves",
            xaxis_title="False Positive Rate",
            yaxis_title="True Positive Rate",
            template='plotly_dark'
        )
        fig5.show()

        # Visualization 6: Pie Chart of Metrics Distribution
        fig6 = go.Figure()
        fig6.add_trace(go.Pie(
            labels=list(metrics.keys()),
            values=list(metrics.values()),
            textinfo='label+percent',
            marker=dict(colors=px.colors.sequential.Plasma)
        ))
        fig6.update_layout(
            title=f"{model_name}: Metrics Distribution",
            template='plotly_dark'
        )
        fig6.show()

        # Visualization 7: Box Plot of Metrics
        fig7 = go.Figure()
        fig7.add_trace(go.Box(
            y=list(metrics.values()),
            x=list(metrics.keys()),
            name=model_name,
            marker_color='purple'
        ))
        fig7.update_layout(
            title=f"{model_name}: Metrics Box Plot",
            xaxis_title="Metric",
            yaxis_title="Score",
            template='plotly_dark'
        )
        fig7.show()

        # Visualization 8: Violin Plot of Metrics
        fig8 = go.Figure()
        fig8.add_trace(go.Violin(
            y=list(metrics.values()),
            x=list(metrics.keys()),
            name=model_name,
            box_visible=True,
            meanline_visible=True,
            fillcolor='lightgreen'
        ))
        fig8.update_layout(
            title=f"{model_name}: Metrics Violin Plot",
            xaxis_title="Metric",
            yaxis_title="Score",
            template='plotly_dark'
        )
        fig8.show()

        # Visualization 9: Scatter Plot of Metrics
        fig9 = go.Figure()
        fig9.add_trace(go.Scatter(
            x=list(metrics.keys()),
            y=list(metrics.values()),
            mode='markers',
            marker=dict(size=15, color='red'),
            name=model_name
        ))
        fig9.update_layout(
            title=f"{model_name}: Metrics Scatter Plot",
            xaxis_title="Metric",
            yaxis_title="Score",
            template='plotly_dark'
        )
        fig9.show()

        # Visualization 10: Area Plot of Metrics
        fig10 = go.Figure()
        fig10.add_trace(go.Scatter(
            x=list(metrics.keys()),
            y=list(metrics.values()),
            fill='tozeroy',
            name=model_name,
            line_color='yellow'
        ))
        fig10.update_layout(
            title=f"{model_name}: Metrics Area Plot",
            xaxis_title="Metric",
            yaxis_title="Score",
            template='plotly_dark'
        )
        fig10.show()

    # Model Comparison Visualizations
    metrics_list = list(next(iter(results.values())).keys())
    fig_comp = make_subplots(
        rows=2, cols=3,
        subplot_titles=metrics_list,
        specs=[[{'type': 'xy'}, {'type': 'xy'}, {'type': 'xy'}],
               [{'type': 'xy'}, {'type': 'xy'}, {'type': 'xy'}]]
    )

    for i, metric in enumerate(metrics_list, 1):
        for model_name in results:
            fig_comp.add_trace(
                go.Bar(
                    x=[model_name],
                    y=[results[model_name][metric]],
                    name=model_name if i == 1 else "",
                    showlegend=(i == 1)
                ),
                row=(i-1)//3 + 1,
                col=(i-1)%3 + 1
            )

    fig_comp.update_layout(
        title="Model Comparison Across Metrics",
        barmode='group',
        height=800,
        template='plotly_dark',
        showlegend=True
    )
    fig_comp.show()

    # Comparison Visualization 2: Radar Plot for All Models
    fig_radar_comp = go.Figure()
    for model_name in results:
        fig_radar_comp.add_trace(go.Scatterpolar(
            r=list(results[model_name].values()),
            theta=list(results[model_name].keys()),
            fill='toself',
            name=model_name
        ))
    fig_radar_comp.update_layout(
        title="Model Comparison: Radar Plot",
        polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
        template='plotly_dark'
    )
    fig_radar_comp.show()

# Main Execution
if __name__ == "__main__":
    print("Starting training process...")
    results, confusion_matrices, roc_curves, best_model, best_model_name = train_and_evaluate()
    print("Training completed. Generating visualizations...")
    visualize_results(results, confusion_matrices, roc_curves)
    print("Visualizations displayed in notebook.")
    print(f"Best model ({best_model_name}) saved as 'best_model.h5' or 'best_model_cnn.h5' and 'best_model_rf.pkl' for DeepLearning+ML.")

Starting training process...
Found 618 images belonging to 4 classes.
Found 152 images belonging to 4 classes.
Training DeepLearning+ML...


Training ResNet+VGG+Inception...


c:\Users\KISHOR\miniconda3\envs\pcos_env\lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 112s 5s/step - accuracy: 0.4067 - loss: 2.7936 - val_accuracy: 0.5658 - val_loss: 0.9766
Epoch 2/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 94s 5s/step - accuracy: 0.6273 - loss: 1.0048 - val_accuracy: 0.7303 - val_loss: 0.7365
Epoch 3/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 95s 5s/step - accuracy: 0.6799 - loss: 0.7958 - val_accuracy: 0.7039 - val_loss: 0.8210
Epoch 4/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 96s 5s/step - accuracy: 0.6836 - loss: 0.8603 - val_accuracy: 0.6579 - val_loss: 0.7149
Epoch 5/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 95s 5s/step - accuracy: 0.7190 - loss: 0.6965 - val_accuracy: 0.7303 - val_loss: 0.6798
Epoch 6/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 92s 5s/step - accuracy: 0.7428 - loss: 0.6533 - val_accuracy: 0.7961 - val_loss: 0.6116
Epoch 7/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 93s 5s/step - accuracy: 0.7460 - loss: 0.6384 - val_accuracy: 0.7500 - val_loss: 0.5984
Epoch 8/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 94s 5s/step - accuracy: 0.7940 - loss: 0.6294 - val_accuracy: 0.8092 - val_loss

Training DenseNet+MobileNet+AlexNet...
Epoch 1/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 63s 2s/step - accuracy: 0.4491 - loss: 1.6580 - val_accuracy: 0.8158 - val_loss: 0.5105
Epoch 2/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 44s 2s/step - accuracy: 0.7387 - loss: 0.7913 - val_accuracy: 0.7895 - val_loss: 0.4906
Epoch 3/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 44s 2s/step - accuracy: 0.8374 - loss: 0.4536 - val_accuracy: 0.8158 - val_loss: 0.4230
Epoch 4/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 44s 2s/step - accuracy: 0.8134 - loss: 0.4668 - val_accuracy: 0.8355 - val_loss: 0.4613
Epoch 5/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 45s 2s/step - accuracy: 0.8313 - loss: 0.4632 - val_accuracy: 0.8750 - val_loss: 0.3757
Epoch 6/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 46s 2s/step - accuracy: 0.8859 - loss: 0.3796 - val_accuracy: 0.8553 - val_loss: 0.4048
Epoch 7/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 46s 2s/step - accuracy: 0.8645 - loss: 0.3635 - val_accuracy: 0.8224 - val_loss: 0.4305
Epoch 8/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 46s 2s/step - accuracy: 0.9005 - loss: 0.

Training completed. Generating visualizations...


Visualizations displayed in notebook.
Best model (DenseNet+MobileNet+AlexNet) saved as 'best_model.h5' or 'best_model_cnn.h5' and 'best_model_rf.pkl' for DeepLearning+ML.
